# Case study: severe-weather event, 13–17 November 2026 in Campania (WRF meteo@uniparthenope)

**Data**: hourly WRF model output from the meteo@uniparthenope, domain **d03**
(Campania, resolution ~3 km), *history* series downloaded from the center's OPeNDAP service with
`scripts/fetch_wrf_d03.py` into the platform read-only *raw* area.

**Pipeline**: raw data → Xarray + Dask (opening ~120 hourly files) → 2 m temperature, 10 m wind,
hourly and cumulative precipitation → city series → curated product Zarr → dashboard Voilà.

The notebook is parameterized (the `parameters` cell) and can be run with Papermill.

In [ ]:
data_dir = "/home/jovyan/data/raw/models/wrf5_d03"
output_dir = "/home/jovyan/work/outputs/wrf_campania"
start = "2026-11-13T00"
end = "2026-11-17T23"
focus_city = "Positano"
time_chunk = 24
n_workers = 2

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import dask
from dask.distributed import Client, LocalCluster

CITIES = {  # approximate city-center coordinates
    "Napoli": (40.852, 14.268), "Salerno": (40.678, 14.765), "Avellino": (40.914, 14.793),
    "Benevento": (41.130, 14.782), "Caserta": (41.072, 14.333), "Positano": (40.628, 14.485),
    "Capri": (40.551, 14.243),
}
timings, t_total = {}, time.perf_counter()
out = Path(output_dir); (out / "figures").mkdir(parents=True, exist_ok=True)
print("xarray", xr.__version__, "| dask", dask.__version__)

## 1. Dask and lazy opening of hourly files

In [ ]:
t0 = time.perf_counter()
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, memory_limit="1GB",
                       dashboard_address=None)
client = Client(cluster)
timings["dask_startup_seconds"] = time.perf_counter() - t0

t0 = time.perf_counter()
files = sorted(Path(data_dir).glob("wrf5_d03_*Z*.nc"))
grid = xr.open_dataset(Path(data_dir) / "wrf5_d03_grid.nc")
ds = xr.open_mfdataset(files, combine="nested", concat_dim="time", parallel=False,
                       chunks={}, data_vars="minimal", coords="minimal", compat="override")
ds = ds.sortby("time").sel(time=slice(start, end)).chunk({"time": time_chunk})
lat, lon = grid.XLAT.values, grid.XLONG.values
land = grid.LANDMASK.values > 0.5
timings["lazy_open_seconds"] = time.perf_counter() - t0

expected_hours = pd.date_range(start, end, freq="h")
missing_hours = sorted(set(expected_hours) - set(pd.to_datetime(ds.time.values)))
print(f"{len(files)} file, {ds.sizes['time']} hours in the period, grid "
      f"{ds.sizes['south_north']}x{ds.sizes['west_east']}, missing hours: {len(missing_hours)}")

## 2. Derived quantities
* 2 m temperature in °C;
* 10 m wind speed: the magnitude of (U10, V10) is independent of WRF grid rotation;
* hourly precipitation: the variables `RAINC` + `RAINNC` are **accumulated from the beginning of each run**
  of the model; the *history* series joins successive runs, so the cumulative field can reset to zero.
  Hourly rain is the difference between consecutive hours; where the difference is negative (a reset)
  use that hour's cumulative value. The number of resets is reported in the results.

In [ ]:
t0 = time.perf_counter()
t2c = ds.T2 - 273.15
ws = np.hypot(ds.U10, ds.V10)
acc = ds.RAINC + ds.RAINNC
dacc = acc.diff("time")
rain_h = xr.where(dacc >= 0, dacc, acc.isel(time=slice(1, None)))
rain_h = rain_h.where(rain_h >= 0, 0)  # protect against spurious values
n_reset = int((dacc.mean(("south_north", "west_east")) < 0).sum().compute())
rain_tot = rain_h.sum("time")

land_mask = xr.DataArray(land, dims=("south_north", "west_east"))
t2_mean_map, ws_max_map, rain_tot_map, t2_land_ts, t2_sea_ts, ws_land_ts, rain_land_ts = dask.compute(
    t2c.mean("time"), ws.max("time"), rain_tot,
    t2c.where(land_mask).mean(("south_north", "west_east")),
    t2c.where(~land_mask).mean(("south_north", "west_east")),
    ws.where(land_mask).mean(("south_north", "west_east")),
    rain_h.where(land_mask).mean(("south_north", "west_east")))
timings["field_computation_seconds"] = time.perf_counter() - t0

## 3. City time series (nearest grid point)

In [ ]:
t0 = time.perf_counter()
def nearest(la, lo):
    d = (lat - la) ** 2 + ((lon - lo) * np.cos(np.deg2rad(la))) ** 2
    return np.unravel_index(np.argmin(d), d.shape)

idx = {c: nearest(*xy) for c, xy in CITIES.items()}
sel = {c: dict(south_north=j, west_east=i) for c, (j, i) in idx.items()}
series = {c: dask.compute(t2c.isel(**s), ws.isel(**s), rain_h.isel(**s)) for c, s in sel.items()}
timings["city_series_seconds"] = time.perf_counter() - t0

def argmax_ll(a):
    j, i = np.unravel_index(np.nanargmax(a), a.shape)
    return round(float(lat[j, i]), 2), round(float(lon[j, i]), 2)

stats = {
    "data": "WRF5 d03 meteo@uniparthenope (history)",
    "period": f"{pd.Timestamp(ds.time.values[0]):%Y-%m-%d %H}:00 - {pd.Timestamp(ds.time.values[-1]):%Y-%m-%d %H}:00",
    "available_hours": int(ds.sizes["time"]), "missing_hours": len(missing_hours),
    "grid": f"{ds.sizes['south_north']}x{ds.sizes['west_east']}",
    "cumulative_resets": n_reset,
    "mean_land_t2_C": round(float(t2_land_ts.mean()), 1),
    "mean_sea_t2_C": round(float(t2_sea_ts.mean()), 1),
    "max_wind_ms": round(float(np.nanmax(ws_max_map)), 1),
    "max_wind_location": argmax_ll(ws_max_map.values),
    "mean_land_cumulative_rain_mm": round(float(rain_tot_map.where(land_mask).mean()), 1),
    "max_cumulative_rain_mm": round(float(np.nanmax(rain_tot_map)), 1),
    "max_cumulative_rain_location": argmax_ll(rain_tot_map.values),
    "max_land_rain_hour": f"{pd.Timestamp(rain_land_ts.idxmax().values):%Y-%m-%d %H}:00",
    "cities": {c: {"mean_t2_C": round(float(s[0].mean()), 1), "min_t2_C": round(float(s[0].min()), 1),
                  "max_t2_C": round(float(s[0].max()), 1), "max_wind_ms": round(float(s[1].max()), 1),
                  "total_rain_mm": round(float(s[2].sum()), 1),
                  "max_hourly_rain_mm": round(float(s[2].max()), 1)}
              for c, s in series.items()},
}
print(json.dumps(stats, indent=2, ensure_ascii=False))

## 4. Visualizations

In [ ]:
t0 = time.perf_counter()
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
def plot_map(field, title, label, cmap, filename, **kw):
    fig, ax = plt.subplots(figsize=(7, 6))
    m = ax.pcolormesh(lon, lat, field, cmap=cmap, shading="auto", **kw)
    ax.contour(lon, lat, land.astype(float), levels=[0.5], colors="k", linewidths=0.6)
    for c, (la, lo) in CITIES.items():
        ax.plot(lo, la, "k.", ms=4); ax.annotate(c, (lo, la), fontsize=7, xytext=(3, 3),
                                                 textcoords="offset points")
    fig.colorbar(m, ax=ax, label=label, shrink=.85)
    ax.set(title=title, xlabel="Longitude [°E]", ylabel="Latitude [°N]"); ax.set_aspect("equal")
    fig.tight_layout(); fig.savefig(out / "figures" / filename, dpi=200); plt.show()

period = stats["period"]
plot_map(rain_tot_map, f"Cumulative precipitation\n{period}", "mm", "Blues", "fig1_cumulative_rain.png")
plot_map(ws_max_map, f"Maximum 10 m wind speed\n{period}", "m/s", "YlOrRd", "fig2_max_wind.png")
plot_map(t2_mean_map, f"Mean 2 m temperature\n{period}", "°C", "RdYlBu_r", "fig3_mean_t2.png")

fig, axs = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
for c in ("Napoli", "Salerno", "Avellino", focus_city):
    s = series[c]
    axs[0].plot(s[0].time, s[0], label=c); axs[1].plot(s[1].time, s[1], label=c)
rf = series[focus_city][2]
axs[2].bar(rf.time.values, rf.values, width=1 / 24, color="tab:blue")
axs[0].set_ylabel("T 2 m [°C]"); axs[1].set_ylabel("10 m wind [m/s]")
axs[2].set_ylabel(f"Hourly rain\n{focus_city} [mm]"); axs[0].legend(ncol=4, fontsize=8)
for a in axs: a.grid(alpha=.3)
fig.suptitle("Hourly city series (nearest grid point)"); fig.tight_layout()
fig.savefig(out / "figures" / "fig4_city_series.png", dpi=200); plt.show()
timings["plotting_seconds"] = time.perf_counter() - t0

## 5. Curated Zarr product

In [ ]:
t0 = time.perf_counter()
curated = xr.Dataset(
    {"t2c": t2c.isel(time=slice(1, None)), "ws10": ws.isel(time=slice(1, None)), "rain_h": rain_h,
     "rain_tot": rain_tot_map, "t2_mean": t2_mean_map, "ws_max": ws_max_map,
     "lat": (("south_north", "west_east"), lat), "lon": (("south_north", "west_east"), lon),
     "landmask": (("south_north", "west_east"), land.astype("int8"))},
    attrs={"title": "13–17 November 2026 Campania event — wxhub curated product",
           "source": "WRF5 d03 meteo@uniparthenope (history), open data IODL 2.0",
           "history": f"created by wrf_campania_case_study.ipynb on {pd.Timestamp.now():%Y-%m-%d}"})
zarr_path = out / "wrf_campania_curated.zarr"
curated.chunk({"time": time_chunk}).to_zarr(zarr_path, mode="w")
chk = xr.open_zarr(zarr_path)
assert np.allclose(chk.rain_tot.values, rain_tot_map.values, equal_nan=True)
timings["zarr_write_seconds"] = time.perf_counter() - t0
size_mb = sum(f.stat().st_size for f in zarr_path.rglob("*") if f.is_file()) / 1e6
print(f"Zarr: {zarr_path} ({size_mb:.1f} MB), verified")

In [ ]:
timings["total_seconds"] = time.perf_counter() - t_total
stats["zarr_MB"] = round(size_mb, 1)
stats["timings_seconds"] = {k: round(v, 2) for k, v in timings.items()}
(out / "results.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
print(json.dumps(stats["timings_seconds"], indent=2))
client.close(); cluster.close()